# 01 - Data Understanding

Notebook ini merupakan tahap **data understanding**, yaitu mengenali struktur, tipe data, dan kualitas dataset IMDb sebelum dilakukan pembersihan dan analisis.

**Tujuan tahap ini:**

1. Mengetahui ukuran dan cakupan data.
2. Memahami arti dan tipe data setiap kolom.
3. Menemukan masalah kualitas data (missing value, duplikat, outlier).
4. Menjadi dasar penentuan langkah *data cleaning* pada notebook berikutnya.

## 1. Import Pandas

Mengimpor library `pandas` yang digunakan untuk mengelola data tabular dalam bentuk **DataFrame**.

In [ ]:
# Mengimpor library pandas untuk manipulasi dan analisis data
import pandas as pd

## 2. Load Dataset

Membaca file `data/imdb_movie_dataset.csv` ke dalam DataFrame bernama `df`.

In [ ]:
# Membaca dataset IMDb dari file CSV ke dalam sebuah DataFrame
df = pd.read_csv("../data/imdb_movie_dataset.csv")

## 3. Melihat Data Awal

Menampilkan **5 baris pertama** untuk memastikan data terbaca benar dan mengenali isi setiap kolom.

In [ ]:
# Menampilkan 5 baris pertama untuk melihat gambaran awal data
df.head()

## 4. Melihat Ukuran Dataset

Hasil `df.shape` adalah **1000 film (baris)** dan **12 kolom (variabel)**.

In [ ]:
# Menampilkan ukuran dataset dalam bentuk (jumlah baris, jumlah kolom)
df.shape

## 5. Melihat Nama Kolom

Menampilkan 12 nama kolom beserta urutannya.

In [ ]:
# Menampilkan nama-nama kolom (fitur) yang ada di dalam dataset
df.columns

## 6. Melihat Tipe Data & Kondisi Kolom

`df.info()` menampilkan jumlah entri, tipe data, jumlah nilai non-null, dan pemakaian memori.

In [ ]:
# Menampilkan ringkasan struktur data (tipe data, jumlah non-null, dan memori)
df.info()

## 7. Melihat Statistik Dasar

Statistik deskriptif (count, mean, std, min, kuartil, max) untuk kolom numerik.

In [ ]:
# Menampilkan statistik deskriptif kolom numerik (count, mean, std, min, kuartil, dan max)
df.describe()

## 8. Mengecek Missing Value

Hasil: **`Revenue (Millions)` = 128 kosong** dan **`Metascore` = 64 kosong**. Kolom lain lengkap terisi.

In [ ]:
# Menghitung jumlah nilai kosong (missing value) pada setiap kolom
df.isnull().sum()

## 9. Mengecek Duplicate

Hasil: **0 baris duplikat**, sehingga tidak diperlukan penghapusan duplikat.

In [ ]:
# Menghitung jumlah baris yang terduplikasi
df.duplicated().sum()

## 10. Mengecek Unique Value

Menghitung jumlah nilai unik tiap kolom untuk mengenali jenis variabel dan potensi masalah.

In [ ]:
# Menghitung jumlah nilai unik pada setiap kolom
df.nunique()

## 11. Memahami Genre

Kolom `Genre` berisi **kombinasi 1-3 genre** dalam satu sel (dipisahkan koma). Berikut eksplorasi untuk memahami komposisinya.

In [ ]:
# Menghitung 20 kombinasi genre yang paling banyak muncul
top_20_genres = (
    df["Genre"]
    .value_counts()
    .head(20)
    .reset_index()
)

# Mengganti nama kolom agar lebih mudah dibaca
top_20_genres.columns = ["Kombinasi Genre", "Jumlah Film"]
top_20_genres

In [ ]:
# Menghitung distribusi jumlah genre per film (1 genre, 2 genre, dst.)
df_genre_count = (
    df["Genre"]
    .str.split(",")   # memisahkan genre yang dipisahkan koma menjadi list
    .str.len()        # menghitung banyaknya genre pada setiap film
    .value_counts()   # menghitung jumlah film untuk tiap jumlah genre
    .sort_index()     # mengurutkan berdasarkan jumlah genre
    .reset_index()    # mengubah hasil Series menjadi DataFrame
)

# Mengganti nama kolom agar sesuai
df_genre_count.columns = ["Jumlah Genre dalam 1 Film", "Jumlah Film"]
df_genre_count

In [ ]:
# Memecah setiap genre menjadi baris terpisah agar bisa dihitung per genre tunggal
genre_exploded = df["Genre"].str.split(",").explode()

# Menghitung jumlah film untuk setiap genre tunggal
genre_exploded.value_counts()

## 12. Mengecek Judul yang Sama

Terdapat 1 judul yang muncul dua kali, yaitu **`The Host`** (2013 - Andrew Niccol dan 2006 - Bong Joon Ho). Keduanya adalah **film yang berbeda**, jadi bukan duplikat.

In [ ]:
# Menampilkan semua baris yang judulnya muncul lebih dari satu kali
# keep=False -> menandai semua baris yang terlibat duplikasi judul
df[df["Title"].duplicated(keep=False)].sort_values("Title")

## 13. Mengecek Nilai Minimum & Maksimum

Melihat rentang nilai untuk `Rating`, `Runtime (Minutes)`, dan `Revenue (Millions)`.

In [ ]:
# Mengecek nilai minimum, median, rata-rata, dan maksimum untuk kolom numerik utama
kolom_numerik = ["Rating", "Runtime (Minutes)", "Revenue (Millions)"]

df[kolom_numerik].agg(["min", "median", "mean", "max"]).T

## 14. Pemeriksaan Awal Outlier

`Revenue (Millions)` memiliki nilai maksimum yang jauh di atas median, sehingga diperiksa dengan metode IQR. Hasilnya ada **55 film** di atas batas atas - pendapatan ini perlu dievaluasi lebih lanjut pada tahap cleaning.

In [ ]:
# Pemeriksaan awal outlier pada Revenue (Millions) dengan metode IQR
Q1 = df["Revenue (Millions)"].quantile(0.25)
Q3 = df["Revenue (Millions)"].quantile(0.75)
IQR = Q3 - Q1

batas_bawah = Q1 - 1.5 * IQR
batas_atas = Q3 + 1.5 * IQR

# Menghitung jumlah film yang pendapatannya berada di luar batas wajar
outlier_revenue = df[
    (df["Revenue (Millions)"] < batas_bawah) | (df["Revenue (Millions)"] > batas_atas)
]

print(f"Batas bawah            : {batas_bawah:.2f}")
print(f"Batas atas             : {batas_atas:.2f}")
print(f"Jumlah outlier Revenue : {outlier_revenue.shape[0]} film")